In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

# level definition
# low, medium, high = 1, 2, 3

In [ ]:
# Prepare data

houses_by_lga = pd.read_csv("Houses by LGA.csv")
houses_by_lga.set_index(["LGA", "Year"], inplace=True)

offences_by_lga = pd.read_csv("Offences By LGA NEW.csv")
offences_by_lga.set_index(["LGA", "Year"], inplace=True)

egm_by_lga = pd.read_csv("EGM-New-Format.csv")
egm_by_lga.set_index(["LGA", "Year"], inplace=True)



merged_data = houses_by_lga.merge(offences_by_lga, how="inner", on=["LGA", "Year"]) \
                           .merge(egm_by_lga, how="inner", on=["LGA", "Year"])

merged_data.rename(columns={"total dollars" : "Total Loss On EGM", "rate" : "EGM Rate"}, inplace=True)

merged_data

In [ ]:
print(merged_data["Weighted House Price"].groupby("Year").mean())
print()
print(merged_data["Weighted House Price"].groupby("Year").quantile(.333))
print()
print(merged_data["Weighted House Price"].groupby("Year").quantile(.667))
print()

In [ ]:
low_weighted_house_price = merged_data["Weighted House Price"].groupby("Year").quantile(.333)
high_weighted_house_price = merged_data["Weighted House Price"].groupby("Year").quantile(.667)

print(low_weighted_house_price.index.get_level_values("Year"))

def weighted_house_price_level_generator(year, price):
    low_price = low_weighted_house_price[low_weighted_house_price.index.get_level_values("Year") == year].values[0]
    high_price = high_weighted_house_price[high_weighted_house_price.index.get_level_values("Year") == year].values[0]

    if price < low_price:
        return "Low"
    elif low_price <= price < high_price:
        return "Medium"
    else:
        return "High"


In [ ]:
merged_data["Weighted House Price Level"] = np.nan

for index, row in merged_data.iterrows():
    merged_data.loc[index, "Weighted House Price Level"] = weighted_house_price_level_generator(
        index[1],
        merged_data.loc[index, "Weighted House Price"]
    )

merged_data

In [ ]:
# Knn

train = pd.concat([
    merged_data[merged_data.index.get_level_values("Year") == 2016],
    merged_data[merged_data.index.get_level_values("Year") == 2018],
    merged_data[merged_data.index.get_level_values("Year") == 2019],
   

])

test = pd.concat([
    merged_data[merged_data.index.get_level_values("Year") == 2017],  
    merged_data[merged_data.index.get_level_values("Year") == 2020],
])

X_COLS = [
    "Offence Rate",
    "EGM Rate"
]

y_COL = ["Weighted House Price Level"]


X_train = train[X_COLS]
y_train = train[y_COL]

X_test = test[X_COLS]
y_test = test[y_COL]

knn = KNeighborsClassifier(n_neighbors=3)


knn.fit(X_train, y_train)

# Calculate accuracy score for the test data
y_pred = knn.predict(X_test)
accuracy = knn.score(X_test, y_test)

accuracy